# TTS Pipeline — Colab GPU Worker

This notebook runs the Abogen TTS engine + pipeline worker on Colab's free GPU.

## Before you start

**On your local machine**, expose Redis so Colab can reach it:
```bash
# Option A: ngrok (recommended)
ngrok tcp 6379
# → copy the  tcp://X.tcp.ngrok.io:XXXXX  URL

# Option B: cloudflared
cloudflared tunnel --url tcp://localhost:6379
```

Then fill in `REDIS_URL` and `WORKER_ID` in the Config cell below.

**MP3 output** goes to Google Drive → your local merger reads it from the same Drive mount.

In [ ]:
# ── 1. Mount Google Drive (MP3 output) ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/tts-pipeline/outputs'
SPOOL_DIR  = '/content/spool'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPOOL_DIR,  exist_ok=True)
print('Drive mounted. Output dir:', OUTPUT_DIR)

In [ ]:
# ── 2. Config — fill these in ─────────────────────────────────────────────────
REDIS_URL  = 'redis://X.tcp.ngrok.io:XXXXX'   # ← your ngrok TCP URL
WORKER_ID  = 'colab-gpu-1'                     # ← unique name for this node
ABOGEN_HOST = 'http://localhost:8808'           # Abogen runs locally on this Colab instance

os.environ['REDIS_URL']   = REDIS_URL
os.environ['WORKER_ID']   = WORKER_ID
os.environ['ABOGEN_HOST'] = ABOGEN_HOST
os.environ['OUTPUT_DIR']  = OUTPUT_DIR
os.environ['SPOOL_DIR']   = SPOOL_DIR
os.environ['USE_GPU']     = 'true'
os.environ['PROMETHEUS_PORT'] = '8000'
print('Config set.')

In [ ]:
# ── 3. Install system deps ────────────────────────────────────────────────────
!apt-get install -y espeak-ng ffmpeg > /dev/null 2>&1
print('espeak-ng + ffmpeg installed.')

In [ ]:
# ── 4. Install Abogen + GPU PyTorch ──────────────────────────────────────────
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!git clone --depth=1 https://github.com/wizardgang/audiobook-stack /content/audiobook-stack 2>/dev/null || \
 git -C /content/audiobook-stack pull
!pip install -q -e '/content/audiobook-stack/abogen_src[gpu-cu121]' 2>/dev/null || \
 pip install -q -e '/content/audiobook-stack/abogen_src'
!pip install -q redis prometheus_client requests
print('All packages installed.')

In [ ]:
# ── 5. Start Abogen TTS server in background ─────────────────────────────────
import subprocess, time

abogen_proc = subprocess.Popen(
    ['python', '-m', 'abogen.webui.app', '--host', '0.0.0.0', '--port', '8808'],
    cwd='/content/audiobook-stack',
    stdout=open('/content/abogen.log', 'w'),
    stderr=subprocess.STDOUT,
)

# Wait for it to be ready
import requests as _req
for i in range(30):
    try:
        r = _req.get('http://localhost:8808/api/voices', timeout=2)
        if r.ok:
            print(f'Abogen ready after {i*2}s')
            break
    except:
        pass
    time.sleep(2)
else:
    print('Abogen did not start — check /content/abogen.log')

In [ ]:
# ── 6. Verify Redis connection ────────────────────────────────────────────────
import redis
r = redis.from_url(REDIS_URL, decode_responses=True, socket_connect_timeout=5)
print('Redis ping:', r.ping())  # should print True
print('TTS queue depth:', r.llen('pipeline:tts'))

In [ ]:
# ── 7. Copy worker.py from repo and run ──────────────────────────────────────
import shutil
shutil.copy('/content/audiobook-stack/worker/worker.py', '/content/worker.py')

# Run worker in background thread so the cell doesn't block
import threading, importlib.util, sys

spec = importlib.util.spec_from_file_location('worker', '/content/worker.py')
worker_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(worker_mod)

t = threading.Thread(target=worker_mod.main, daemon=True, name='pipeline-worker')
t.start()
print(f'Worker {WORKER_ID} started — consuming pipeline:tts from {REDIS_URL}')

In [ ]:
# ── 8. Keep-alive + live status (run this cell last) ─────────────────────────
# Colab disconnects idle runtimes — this loop keeps it alive and shows progress.
import time

r2 = redis.from_url(REDIS_URL, decode_responses=True)
while True:
    try:
        queue_depth = r2.llen('pipeline:tts')
        heartbeat   = r2.hget('worker:heartbeats', WORKER_ID) or '0'
        age = int(time.time()) - int(heartbeat)
        print(f'\r[{time.strftime("%H:%M:%S")}] queue={queue_depth} | {WORKER_ID} last_seen={age}s ago   ', end='')
    except Exception as e:
        print(f'\nRedis error: {e}')
    time.sleep(10)